In [6]:
#Because sample size is so small I will use Gemini to analyze the sentiment
import os
from pathlib import Path
import sys
from google import genai
from google.genai import types

parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))
from config import GEMINI_API_KEY
#Initialize the Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)




In [13]:
import re
def evaluate_sentiment(text: str) -> float:
    # Create a request for the Gemini API
    prompt = f"""
        This text is pulled from the nascar subreddit. It may or may not discuss sentiment toward one of these sponsors: FedEx, Cheddar's Scratch Kitchen, Love's Travel Stops, Busch Light, or Castrol.

        Analyze the sentiment expressed toward the sponsor in the text below. Respond with ONLY a single number between -1 and 1, where -1 is very negative, 0 is neutral, and 1 is very positive. Do not include any explanation, words, or punctuation — output the number only.

        Text: {text}
        """
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=10,
            thinking_config=types.ThinkingConfig(thinking_budget=0)
        )
    )
    assert response.text is not None, "No response from Gemini API"
    raw = response.text.strip()
    assert raw != "", "Empty response from Gemini API"
    match = re.search(r"-?\d+\.?\d*", raw)
    assert match, f"Could not parse a number from response: {raw!r}"
    return float(match.group())
#Test Use
test_text = "FedEx driver so buns! I can't watch this crap"
result = evaluate_sentiment(test_text)
print(f"Sentiment score for test text: {result}")

Sentiment score for test text: -0.8
